<a href="https://colab.research.google.com/github/joe-singh/su2diffusion/blob/main/SU(2)DiffusionHeatKernel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Heat Kernel $SU(2)$ Distribution

## Install the branch

In [ ]:
# For PR experiments, replace main with the branch name, e.g. codex/package-workflow.
!pip install -q git+https://github.com/joe-singh/su2diffusion.git@main

## Imports

In [ ]:
import matplotlib.pyplot as plt
import torch
from IPython.display import HTML

from su2diffusion import (
    DiffusionSchedule,
    TrainConfig,
    diagnose_samples,
    print_diagnostics_table,
    train_heat_kernel_model,
)
from su2diffusion.data import default_centers, sample_clean_blobs
from su2diffusion.device import get_default_device
from su2diffusion.diffusion import brownian_forward_heat_target
from su2diffusion.quaternion import sample_haar
from su2diffusion.sampling import sample_reverse, sample_reverse_trajectory
from su2diffusion.viz import (
    animate_reverse_histogram,
    plot_nearest_center_histogram,
    plot_quaternions,
)

device = get_default_device()
print("device:", device)
torch.manual_seed(0)

## Params

In [ ]:
schedule = DiffusionSchedule(T=200, beta_start=1e-4, beta_end=0.005)
train_config = TrainConfig(batch_size=2048, num_steps=2000, hidden=512, n_terms=128)

betas, sigma2, sigmas = schedule.tensors(device)
print("sigma_T^2:", sigma2[-1].item())
print("sigma_T:", sigmas[-1].item())

## Training

In [ ]:
model, losses = train_heat_kernel_model(
    train_config=train_config,
    schedule=schedule,
    device=device,
)

plt.figure(figsize=(6, 4))
plt.plot(losses)
plt.yscale("log")
plt.xlabel("training step")
plt.ylabel("MSE loss")
plt.title("Heat-kernel target training loss")
plt.show()

## Forward Noising

In [ ]:
with torch.no_grad():
    q0_plot, labels_plot = sample_clean_blobs(5000, device=device)

    t_mid = torch.full((5000,), schedule.T // 2, device=device, dtype=torch.long)
    qt_mid, _ = brownian_forward_heat_target(q0_plot, t_mid, schedule=schedule)

    t_final = torch.full((5000,), schedule.T, device=device, dtype=torch.long)
    qt_final, _ = brownian_forward_heat_target(q0_plot, t_final, schedule=schedule)

plot_quaternions(q0_plot, "Clean SU(2) blobs", labels_plot)
plot_quaternions(qt_mid, "Brownian noised blobs: middle timestep", labels_plot)
plot_quaternions(qt_final, "Brownian noised blobs: final timestep", labels_plot)

## Reverse Sampling

In [ ]:
eta = 0.7
centers = default_centers(device=device)

with torch.no_grad():
    q_clean, _ = sample_clean_blobs(5000, centers=centers)
    q_haar = sample_haar(5000, device=device)
    q_gen_det = sample_reverse(model, schedule, n_samples=5000, eta=0.0, device=device)
    q_gen_sto = sample_reverse(model, schedule, n_samples=5000, eta=eta, device=device)

plot_nearest_center_histogram(
    q_clean,
    q_haar,
    {
        "Generated deterministic": q_gen_det,
        f"Generated stochastic eta={eta}": q_gen_sto,
    },
    centers=centers,
)

diagnostics = {
    "deterministic": diagnose_samples(q_gen_det, q_clean, q_haar, centers=centers),
    "stochastic": diagnose_samples(q_gen_sto, q_clean, q_haar, centers=centers),
}
print_diagnostics_table(diagnostics)
print("nearest-center mass:")
for name, diag in diagnostics.items():
    print(name, [round(x, 3) for x in diag.nearest_center_mass])

## Reverse Trajectory Animation

In [ ]:
q_final, frames, t_values = sample_reverse_trajectory(
    model,
    schedule,
    n_samples=3000,
    eta=eta,
    record_every=2,
    device=device,
)

anim_hist = animate_reverse_histogram(
    frames,
    t_values,
    clean_reference=q_clean.detach().cpu(),
    haar_reference=q_haar.detach().cpu(),
    device=device,
)

HTML(anim_hist.to_jshtml())